# Orpheus Engine Project Reorganization

This notebook provides tools to restructure the Orpheus Engine project according to the HP AI Studio Blueprints Repository format. This ensures consistency, readability, and scalability across all projects in the repository.

## Overview

The reorganization will:
1. Create a standardized directory structure following HP AI Studio guidelines
2. Generate template files for documentation and configuration
3. Relocate existing files to their new locations
4. Validate the resulting structure

Let's begin by importing the required libraries and setting up our workspace.

In [ ]:
# Import required libraries
import os
import sys
import shutil
import pathlib
from pathlib import Path
import json
import yaml
import logging
from datetime import datetime

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('orpheus-reorganization')

# Define root project directory
ROOT_DIR = Path('/Volumes/PRO-BLADE/Github/orpheus-engine')
if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Root directory not found at {ROOT_DIR}")

print(f"Project root directory: {ROOT_DIR}")
print(f"Current working directory: {os.getcwd()}")

## Define HP AI Studio Blueprints Repository Structure

First, we'll define the target directory structure according to the HP AI Studio Blueprints Repository format. For the Orpheus Engine project, we'll organize it under the 'audio-processing' category since it's an audio workstation with AI capabilities.

In [ ]:
# Define the target directory structure for Orpheus Engine
structure = {
    "audio-processing": {  # Category
        "orpheus-engine": {  # Project name
            "configs": [
                "config.yaml",
                # "secrets.yaml" - gitignored, will create template
            ],
            "data": [
                # Sample data files will be populated from existing project
                ".gitkeep"
            ],
            "demo": [
                # Demo files will be populated from existing demo folder
                ".gitkeep"
            ],
            "docs": [
                # Documentation files will be populated from existing docs
                "installation_guide.md",
                "architecture.md"
            ],
            "notebooks": [
                # Notebooks will be populated from existing notebooks
                ".gitkeep"
            ],
            "src": {
                "__init__.py": "",
                "audio": {
                    "__init__.py": "",
                },
                "contexts": {
                    "__init__.py": "",
                },
                "electron": {
                    "__init__.py": "",
                },
                "models": {
                    "__init__.py": "",
                },
                "screens": {
                    "__init__.py": "",
                },
                "services": {
                    "__init__.py": "",
                },
                "utils": {
                    "__init__.py": "",
                },
            },
            ".gitignore": "",
            "requirements.txt": "",
            "README.md": ""
        },
        "README.md": ""  # Category README
    },
    ".gitignore": "",  # Root-level .gitignore
    "LICENSE": "",     # Project license
    "README.md": ""    # Root-level README
}

def print_structure(struct, indent=0):
    """Pretty print the directory structure"""
    for key, value in struct.items():
        if isinstance(value, dict):
            print("  " * indent + f"├── {key}/")
            print_structure(value, indent + 1)
        elif isinstance(value, list):
            print("  " * indent + f"├── {key}/")
            for item in value:
                print("  " * (indent + 1) + f"├── {item}")
        else:
            print("  " * indent + f"├── {key}")

# Display the directory structure
print("Planned Directory Structure:")
print_structure(structure)

## Create Directory Structure

Now, let's implement functions to create the directory structure defined above. We'll create a backup of the existing project first as a safety measure.

In [ ]:
# Create a backup of the existing project
def create_backup(root_dir):
    """Create a tar.gz backup of the current project structure"""
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    backup_name = f"orpheus-backup-{timestamp}.tar.gz"
    backup_dir = root_dir / "backups"
    
    if not backup_dir.exists():
        backup_dir.mkdir(parents=True, exist_ok=True)
        
    backup_path = backup_dir / backup_name
    
    logger.info(f"Creating backup at {backup_path}")
    
    # Use shutil to create a tar.gz archive
    # Exclude large directories and unnecessary files
    excluded_dirs = ["node_modules", ".git", "backups", "archives"]
    
    def filter_function(path):
        for excluded in excluded_dirs:
            if excluded in str(path):
                return False
        return True
    
    try:
        shutil.make_archive(
            str(backup_path).replace('.tar.gz', ''),
            'gztar',
            root_dir=root_dir,
            base_dir='.',
            logger=logger,
            verbose=1,
        )
        logger.info(f"Backup created successfully at {backup_path}")
        return backup_path
    except Exception as e:
        logger.error(f"Failed to create backup: {e}")
        return None

# Run the backup process - uncomment when ready to execute
# backup_path = create_backup(ROOT_DIR)

In [ ]:
# Functions to create the directory structure
def create_directory_structure(root_dir, structure, base_path=None):
    """Recursively create directory structure from the structure dictionary"""
    if base_path is None:
        base_path = root_dir
    
    for key, value in structure.items():
        path = base_path / key
        
        # If value is a dictionary, it's a directory
        if isinstance(value, dict):
            path.mkdir(parents=True, exist_ok=True)
            logger.info(f"Created directory: {path}")
            create_directory_structure(root_dir, value, path)
        
        # If value is a list, it's a directory with files
        elif isinstance(value, list):
            path.mkdir(parents=True, exist_ok=True)
            logger.info(f"Created directory: {path}")
            
            for file_name in value:
                file_path = path / file_name
                if not file_path.exists():
                    file_path.touch()
                    logger.info(f"Created file: {file_path}")
        
        # If value is a string, it's a file
        else:
            # Create parent directory if needed
            if not path.parent.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                
            # Create the file with content if provided
            if not path.exists():
                if value:
                    with open(path, 'w') as f:
                        f.write(value)
                else:
                    path.touch()
                logger.info(f"Created file: {path}")

# Function to create temporary staging directory for reorganization
def create_staging_area(root_dir):
    """Create a staging area for the reorganization"""
    staging_dir = root_dir / "_staging_hp_ai_studio"
    
    if staging_dir.exists():
        logger.warning(f"Staging directory already exists at {staging_dir}")
        response = input("Do you want to remove the existing staging directory? (y/n): ")
        if response.lower() == 'y':
            shutil.rmtree(staging_dir)
            logger.info(f"Removed existing staging directory: {staging_dir}")
        else:
            logger.info("Using existing staging directory")
            return staging_dir
    
    staging_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Created staging directory: {staging_dir}")
    
    return staging_dir

# Create staging area - uncomment when ready to execute
# staging_dir = create_staging_area(ROOT_DIR)
# print(f"Staging directory created at: {staging_dir}")

## Generate Template Files

Next, we'll create the template content for essential files like README.md, .gitignore, and configuration files.

In [ ]:
# Create template file content
def get_root_readme_template():
    """Generate content for the root README.md"""
    return """# Orpheus Engine

A digital audio workstation (DAW) with AI-powered audio analysis and processing capabilities.

## Categories

- [Audio Processing](/audio-processing) - Audio processing tools and applications

## Repository Structure

This repository follows the HP AI Studio Blueprints Repository structure:

```
├── category/                           # Folder categorizing projects
│   │
│   ├── sample-project/                 # Specific sample project folder
│   │   │
│   │   ├── configs/                    # Configuration files
│   │   ├── data/                       # Data assets used in the project
│   │   ├── demo/                       # UI-related files
│   │   ├── docs/                       # Project documentation
│   │   ├── notebooks/                  # Jupyter notebooks
│   │   ├── src/                        # Core Python modules
│   │   ├── .gitignore                  # Project-specific gitignore
│   │   ├── requirements.txt            # Python dependencies
│   │   └── README.md                   # Project documentation
│   │
│   └── README.md                       # Category overview
│
├── .gitignore                          # Root-level .gitignore
├── LICENSE                             # Project license
└── README.md                           # This file
```

## License

This project is licensed under the terms of the MIT license. See [LICENSE](LICENSE) for details.

---

> Built with ❤️ using [**HP AI Studio**](https://www.hp.com/us-en/workstations/ai-studio.html).
"""

def get_category_readme_template():
    """Generate content for the category README.md"""
    return """# Audio Processing

Tools and applications for audio processing, analysis, and digital audio workstation (DAW) capabilities.

## Projects

- [Orpheus Engine](/audio-processing/orpheus-engine) - Digital Audio Workstation with AI-powered audio analysis

---

> Built with ❤️ using [**HP AI Studio**](https://www.hp.com/us-en/workstations/ai-studio.html).
"""

def get_project_readme_template():
    """Generate content for the project README.md"""
    return """# Orpheus Engine

![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)
![Python: 3.8+](https://img.shields.io/badge/python-3.8+-blue.svg)
![Node.js: 16+](https://img.shields.io/badge/node.js-16+-green.svg)

## Overview

Orpheus Engine is a digital audio workstation (DAW) with AI-powered audio analysis capabilities. It provides a modern interface for recording, editing, and analyzing audio with machine learning tools integrated directly into the workflow.

## Project Structure

```
├── configs/                  # Configuration files
├── data/                     # Audio samples and training data
├── demo/                     # Demo applications and UIs
├── docs/                     # Documentation
├── notebooks/                # Jupyter notebooks for audio analysis
├── src/                      # Source code
│   ├── audio/                # Audio processing modules
│   ├── contexts/             # React contexts for state management
│   ├── electron/             # Electron application code
│   ├── models/               # ML model definitions
│   ├── screens/              # UI screens and components
│   ├── services/             # Backend services
│   └── utils/                # Utility functions
├── .gitignore                # Git ignore file
├── requirements.txt          # Python dependencies
└── README.md                 # This file
```

## Setup

### Step 0: Hardware Requirements

- **OS**: Windows 10/11, macOS 10.15+, or Linux (Ubuntu 18.04+)
- **CPU**: Intel i5 or AMD Ryzen 5 (2.0GHz+)
- **RAM**: 8GB minimum, 16GB recommended
- **Storage**: 2GB free space
- **GPU**: NVIDIA GPU with CUDA support (optional, for ML acceleration)

### Quick Setup

```bash
# Clone repository
git clone https://github.com/your-username/orpheus-engine.git
cd orpheus-engine

# Install Node.js dependencies
npm install

# Install Python dependencies for AI features
pip install -r requirements.txt

# Start the application
npm run dev
```

## Usage

### Running the DAW

1. Start the application:
   ```bash
   npm run dev
   ```

2. Use the interface to:
   - Create new tracks
   - Record audio
   - Add effects
   - Analyze audio with ML tools

### Running Notebooks

Jupyter notebooks for audio analysis are available in the `notebooks/` directory:

```bash
jupyter notebook notebooks/
```

## Support & Troubleshooting

For help and questions:
- Submit issues on GitHub
- Check documentation in the `docs/` folder

Common fixes:
- Clear Node.js cache: `npm cache clean --force`
- Reinstall dependencies: `rm -rf node_modules && npm install`
- Update Python packages: `pip install -r requirements.txt --upgrade`

---

> Built with ❤️ using [**HP AI Studio**](https://www.hp.com/us-en/workstations/ai-studio.html).
"""

def get_gitignore_template():
    """Generate content for .gitignore"""
    return """# Node.js
node_modules/
npm-debug.log
yarn-debug.log
yarn-error.log
package-lock.json

# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
env/
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
*.egg-info/
.installed.cfg
*.egg

# Jupyter Notebook
.ipynb_checkpoints

# Environment
.env
.venv
env/
venv/
ENV/
.env.local
.env.development.local
.env.test.local
.env.production.local

# VSCode
.vscode/*
!.vscode/settings.json
!.vscode/tasks.json
!.vscode/launch.json
!.vscode/extensions.json

# Logs
logs
*.log

# OS specific
.DS_Store
.DS_Store?
._*
.Spotlight-V100
.Trashes
ehthumbs.db
Thumbs.db

# Sensitive information
configs/secrets.yaml

# ML related
*.h5
*.pt
*.pth
*.tflite
*.onnx
model_checkpoints/
mlruns/

# Audio files
*.wav
*.mp3
*.ogg
*.flac
*.aif
*.aiff
# Whitelist sample data
!data/samples/*.wav
!data/samples/*.mp3
"""

def get_config_template():
    """Generate content for config.yaml"""
    return """# Orpheus Engine Configuration

# Application Settings
application:
  name: Orpheus Engine
  version: 1.0.0
  environment: development  # development, production

# Audio Settings
audio:
  sampleRate: 48000
  bitDepth: 24
  channels: 2
  bufferSize: 1024
  latencyHint: interactive  # interactive, balanced, playback

# User Interface
ui:
  theme: dark  # dark, light
  language: en
  zoomLevel: 1.0
  showTips: true

# Machine Learning
ml:
  modelsPath: ./models
  mlflowTrackingUri: http://localhost:5002
  mlflowExperimentName: orpheus-audio-analysis
  mlflowArtifactRoot: ./mlruns
  useCuda: auto  # auto, true, false

# Development
development:
  debugMode: false
  logLevel: info  # debug, info, warning, error
  hotReload: true
  devServer:
    port: 5173
"""

def get_secrets_template():
    """Generate content for secrets.yaml"""
    return """# Secrets Configuration - DO NOT COMMIT THIS FILE

# API Keys
api:
  huggingface_token: "YOUR_HUGGINGFACE_TOKEN"
  openai_api_key: "YOUR_OPENAI_API_KEY"

# Database
database:
  username: "db_username"
  password: "db_password"
  host: "localhost"
  port: 5432
  name: "orpheus_db"

# Services
services:
  auth_secret: "your_auth_secret_key"
"""

def get_installation_guide_template():
    """Generate content for installation_guide.md"""
    return """# Orpheus Engine - Installation Guide

This guide provides step-by-step instructions for setting up Orpheus Engine on different platforms.

## Prerequisites

- Node.js v16+ (v18+ recommended)
- Python 3.8+ (for AI/ML features)
- Git

## Installation Steps

### 1. Clone the Repository

```bash
git clone https://github.com/your-username/orpheus-engine.git
cd orpheus-engine
```

### 2. Install Node.js Dependencies

```bash
npm install
```

### 3. Install Python Dependencies (for AI Features)

```bash
pip install -r requirements.txt
```

### 4. Configure the Application

Create a local configuration file:

```bash
cp configs/config.yaml configs/config.local.yaml
```

Edit `configs/config.local.yaml` to match your environment.

### 5. Start the Application

```bash
npm run dev
```

## Platform-Specific Instructions

### macOS

Install Homebrew and required dependencies:

```bash
# Install Homebrew (if not already installed)
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

# Install Xcode Command Line Tools
xcode-select --install
```

### Windows

- Install [Visual Studio Build Tools](https://visualstudio.microsoft.com/visual-cpp-build-tools/)
- Install [Windows SDK](https://developer.microsoft.com/en-us/windows/downloads/windows-sdk/)

### Linux (Ubuntu/Debian)

Install system dependencies:

```bash
sudo apt update
sudo apt install -y build-essential python3-dev
sudo apt install -y libasound2-dev  # For audio support
sudo apt install -y dbus-x11 xvfb   # For headless operation
```

## Verifying Installation

To verify that Orpheus Engine is installed correctly:

```bash
# Check Node.js installation
node --version  # Should be v16+
npm --version   # Should be v7+

# Check Python installation (if using AI features)
python --version  # Should be 3.8+
pip --version

# Verify application starts
npm run dev
```

The application should open in a new Electron window.

## Troubleshooting

If you encounter issues during installation:

### Node.js Issues

```bash
# Clear npm cache
npm cache clean --force

# Reinstall node_modules
rm -rf node_modules package-lock.json
npm install
```

### Python/MLFlow Issues

```bash
# Update pip
python -m pip install --upgrade pip

# Install with verbose output
pip install -v mlflow

# Check Python path
which python
which pip
```

### Electron Issues

```bash
# Rebuild Electron
npm run setup:electron

# Clear Electron cache
rm -rf ~/.cache/electron
```

## Need Help?

If you continue to have installation problems:

- Check the [documentation](./README.md)
- Submit an issue on GitHub
- Contact our support team

---

> Built with ❤️ using [**HP AI Studio**](https://www.hp.com/us-en/workstations/ai-studio.html).
"""

def get_architecture_template():
    """Generate content for architecture.md"""
    return """# Orpheus Engine - Architecture Overview

This document provides an overview of the Orpheus Engine's architecture, components, and workflows.

## System Architecture

Orpheus Engine is built as an Electron application with a React frontend and Python backend for AI/ML features. The system architecture consists of several key components:

```
┌─────────────────────────────────────────────┐
│                                             │
│               Electron App                  │
│                                             │
├───────────────────┬─────────────────────────┤
│                   │                         │
│   React Frontend  │   Python Backend        │
│                   │                         │
├───────────────┬───┼─────────────────────────┤
│               │   │                         │
│  Web Audio    │   │   MLflow Integration    │
│    API        │   │                         │
│               │   │                         │
└───────────────┴───┴─────────────────────────┘
```

### Main Components

1. **Electron Application**: Cross-platform desktop application wrapper
2. **React Frontend**: User interface and application logic
3. **Web Audio API**: Audio recording, playback, and processing
4. **Python Backend**: Machine learning and audio analysis
5. **MLflow Integration**: Experiment tracking and model management

## Component Details

### Frontend Architecture

The frontend follows a component-based architecture using React with TypeScript:

```
src/
├── contexts/             # React contexts for state management
│   ├── WorkstationContext.tsx
│   └── TransportContext.tsx
├── screens/              # UI screens and components
│   ├── workstation/      # Main DAW interface
│   └── settings/         # Application settings
├── services/             # Frontend services
│   ├── audioRecording.ts
│   └── fileManagement.ts
└── utils/                # Utility functions
    ├── audioProcessing.ts
    └── timeFormat.ts
```

### Backend Architecture

The Python backend provides AI/ML capabilities:

```
src/
├── audio/                # Audio processing modules
│   ├── analysis.py
│   └── features.py
├── models/               # ML model definitions
│   ├── classifier.py
│   └── segmentation.py
└── services/             # Backend services
    ├── mlflow_service.py
    └── prediction_service.py
```

## Data Flow

1. **Audio Recording**:
   - User initiates recording via UI
   - Web Audio API captures audio data
   - Audio buffer is processed and stored

2. **Audio Analysis**:
   - Recorded audio is sent to Python backend
   - ML models analyze audio features
   - Results are returned to frontend for display

3. **Experiment Tracking**:
   - Analysis parameters and results logged to MLflow
   - Experiments can be compared and visualized
   - Models can be versioned and deployed

## Technology Stack

- **Frontend**:
  - React 18+
  - TypeScript 5+
  - Web Audio API
  - Material-UI

- **Backend**:
  - Python 3.8+
  - MLflow 2.0+
  - librosa (audio processing)
  - scikit-learn / TensorFlow / PyTorch

- **Desktop Application**:
  - Electron 25+
  - Node.js 16+

## Development Workflow

1. **Local Development**:
   - Run `npm run dev` to start the dev environment
   - Frontend changes hot reload in Electron
   - Python backend runs as separate process

2. **Testing**:
   - Frontend tests with Vitest
   - Backend tests with pytest
   - E2E tests with Playwright

3. **Deployment**:
   - Build with `npm run build`
   - Package with Electron Builder
   - Distribute as platform-specific installers

## Integration Points

- **Web Audio API ↔ React**: Audio recording, playback, visualization
- **React ↔ Python**: Data exchange for audio analysis
- **Python ↔ MLflow**: Experiment tracking and model management
- **Electron ↔ OS**: File system access, native menus, notifications

---

> Built with ❤️ using [**HP AI Studio**](https://www.hp.com/us-en/workstations/ai-studio.html).
"""

# Display examples of templates
print("Root README.md template (excerpt):")
print("-" * 80)
print("\n".join(get_root_readme_template().split("\n")[:10]) + "\n...")
print("\nProject README.md template (excerpt):")
print("-" * 80)
print("\n".join(get_project_readme_template().split("\n")[:10]) + "\n...")

## Validate Directory Structure

Let's create functions to validate the directory structure after creation. This will ensure that all required files and directories are present according to the HP AI Studio Blueprints Repository format.

In [ ]:
# Functions to validate directory structure
def validate_structure(base_path, expected_structure):
    """Validate that the created structure matches the expected structure"""
    issues = []
    
    # Check if base_path exists
    if not base_path.exists():
        issues.append(f"Base path {base_path} does not exist")
        return issues
    
    # Validate the structure recursively
    for key, value in expected_structure.items():
        path = base_path / key
        
        # Check if path exists
        if not path.exists():
            issues.append(f"Path {path} does not exist")
            continue
            
        # If value is a dictionary, it's a directory
        if isinstance(value, dict):
            if not path.is_dir():
                issues.append(f"{path} should be a directory but is not")
            else:
                # Recursively validate subdirectories
                sub_issues = validate_structure(path, value)
                issues.extend(sub_issues)
                
        # If value is a list, it's a directory with files
        elif isinstance(value, list):
            if not path.is_dir():
                issues.append(f"{path} should be a directory but is not")
            else:
                # Check that all expected files exist
                for file_name in value:
                    file_path = path / file_name
                    if not file_path.exists():
                        issues.append(f"File {file_path} does not exist")
                        
        # If value is a string, it's a file
        else:
            if not path.is_file():
                issues.append(f"{path} should be a file but is not")
    
    return issues

def print_validation_results(issues):
    """Print the validation results in a readable format"""
    if not issues:
        print("✅ Validation successful! No issues found.")
        return
    
    print(f"❌ Validation found {len(issues)} issues:")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")

# Define validation function for actual execution
def validate_directory_structure(staging_dir, structure):
    """Validate the directory structure in the staging area"""
    print(f"Validating directory structure in {staging_dir}...")
    issues = validate_structure(staging_dir, structure)
    print_validation_results(issues)
    return len(issues) == 0

# This will be run later after creating the structure
# is_valid = validate_directory_structure(staging_dir, structure)

## Create Sample Project Structure

Now, let's demonstrate how to create a complete sample project structure according to the HP AI Studio specifications. We'll implement a function to set up the entire structure in a staging area, and then create a plan for migrating existing files to their new locations.

In [ ]:
# Function to set up complete project structure
def setup_project_structure(root_dir, structure):
    """Set up the complete project structure in the staging area"""
    # Create staging directory
    staging_dir = create_staging_area(root_dir)
    
    # Create directory structure
    create_directory_structure(staging_dir, structure)
    
    # Add template content to important files
    
    # Root level files
    with open(staging_dir / "README.md", "w") as f:
        f.write(get_root_readme_template())
    
    with open(staging_dir / ".gitignore", "w") as f:
        f.write(get_gitignore_template())
        
    # Copy existing LICENSE file if it exists
    if (root_dir / "LICENSE").exists():
        shutil.copy(root_dir / "LICENSE", staging_dir / "LICENSE")
    else:
        # Create a basic MIT license file
        with open(staging_dir / "LICENSE", "w") as f:
            f.write("MIT License placeholder - Add your license text here")
    
    # Category level files
    category_dir = staging_dir / "audio-processing"
    with open(category_dir / "README.md", "w") as f:
        f.write(get_category_readme_template())
    
    # Project level files
    project_dir = category_dir / "orpheus-engine"
    with open(project_dir / "README.md", "w") as f:
        f.write(get_project_readme_template())
    
    with open(project_dir / ".gitignore", "w") as f:
        f.write(get_gitignore_template())
    
    # Copy existing requirements.txt if it exists
    if (root_dir / "requirements.txt").exists():
        shutil.copy(root_dir / "requirements.txt", project_dir / "requirements.txt")
    
    # Config files
    config_dir = project_dir / "configs"
    with open(config_dir / "config.yaml", "w") as f:
        f.write(get_config_template())
    
    with open(config_dir / "secrets.yaml", "w") as f:
        f.write(get_secrets_template())
    
    # Documentation files
    docs_dir = project_dir / "docs"
    with open(docs_dir / "installation_guide.md", "w") as f:
        f.write(get_installation_guide_template())
    
    with open(docs_dir / "architecture.md", "w") as f:
        f.write(get_architecture_template())
    
    # Validate the created structure
    is_valid = validate_directory_structure(staging_dir, structure)
    
    return staging_dir, is_valid

# Define function to create file migration plan
def create_migration_plan(root_dir, staging_dir):
    """Create a plan for migrating existing files to the new structure"""
    migration_plan = []
    
    # Define mappings of source directories to target directories
    mappings = {
        "src": "audio-processing/orpheus-engine/src",
        "demo": "audio-processing/orpheus-engine/demo",
        "docs": "audio-processing/orpheus-engine/docs",
        "notebooks": "audio-processing/orpheus-engine/notebooks",
        "data": "audio-processing/orpheus-engine/data",
    }
    
    # Create migration tasks for each mapping
    for src_dir, target_dir in mappings.items():
        source_path = root_dir / src_dir
        if not source_path.exists() or not source_path.is_dir():
            migration_plan.append({
                "action": "skip",
                "source": str(source_path),
                "reason": "Source directory does not exist"
            })
            continue
            
        target_path = staging_dir / target_dir
        if not target_path.exists():
            target_path.mkdir(parents=True, exist_ok=True)
            
        migration_plan.append({
            "action": "copy_dir",
            "source": str(source_path),
            "target": str(target_path),
            "description": f"Copy {src_dir} directory to new location"
        })
    
    # Handle specific important files
    important_files = {
        "README.md": "audio-processing/orpheus-engine/README.md",
        "requirements.txt": "audio-processing/orpheus-engine/requirements.txt",
        "LICENSE": "LICENSE"
    }
    
    for src_file, target_file in important_files.items():
        source_path = root_dir / src_file
        if not source_path.exists() or not source_path.is_file():
            migration_plan.append({
                "action": "skip",
                "source": str(source_path),
                "reason": "Source file does not exist"
            })
            continue
            
        target_path = staging_dir / target_file
        migration_plan.append({
            "action": "copy_file",
            "source": str(source_path),
            "target": str(target_path),
            "description": f"Copy {src_file} to new location"
        })
    
    return migration_plan

# Create shell script to execute the migration plan
def generate_migration_script(migration_plan, output_path):
    """Generate a shell script to execute the migration plan"""
    script_content = """#!/bin/bash
# Migration script for Orpheus Engine repository reorganization
# Generated on {date}

set -e

echo "Starting Orpheus Engine repository reorganization..."

""".format(date=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

    for task in migration_plan:
        if task["action"] == "skip":
            script_content += f"echo \"Skipping {task['source']}: {task['reason']}\"\n\n"
        elif task["action"] == "copy_dir":
            script_content += f"echo \"Copying directory {task['source']} to {task['target']}\"\n"
            script_content += f"mkdir -p \"{task['target']}\"\n"
            script_content += f"cp -r \"{task['source']}\"/* \"{task['target']}/\"\n\n"
        elif task["action"] == "copy_file":
            script_content += f"echo \"Copying file {task['source']} to {task['target']}\"\n"
            script_content += f"mkdir -p \"$(dirname \"{task['target']}\")\"\n"
            script_content += f"cp \"{task['source']}\" \"{task['target']}\"\n\n"
    
    script_content += """
echo "Repository reorganization complete!"
echo "Please review the new structure in the staging directory."
"""

    with open(output_path, "w") as f:
        f.write(script_content)
    
    # Make the script executable
    os.chmod(output_path, 0o755)
    
    return output_path

# This cell demonstrates the functions, but doesn't execute them yet
print("Functions defined for:")
print("1. Setting up project structure")
print("2. Creating migration plan")
print("3. Generating migration script")
print("\nRun the next cell to execute these functions")

## Execute Reorganization

Now we can execute the reorganization process. This involves several steps:

1. Create a backup of the current project
2. Set up the new structure in a staging area
3. Create a migration plan
4. Generate a migration script
5. Validate the new structure

**Note:** The actual code execution is commented out to prevent accidental execution. Remove the comments when you're ready to proceed.

In [ ]:
# CAUTION: This cell will reorganize your project structure
# Remove the triple quotes to execute

"""
# Step 1: Create a backup of the current project
backup_path = create_backup(ROOT_DIR)
if backup_path is None:
    print("❌ Failed to create backup. Aborting reorganization.")
else:
    print(f"✅ Backup created at {backup_path}")
    
    # Step 2: Set up the new structure in a staging area
    print("\nSetting up new structure in staging area...")
    staging_dir, is_valid = setup_project_structure(ROOT_DIR, structure)
    
    if not is_valid:
        print("❌ Structure validation failed. Please check the issues above.")
    else:
        print(f"✅ Structure created successfully in {staging_dir}")
        
        # Step 3: Create a migration plan
        print("\nCreating migration plan...")
        migration_plan = create_migration_plan(ROOT_DIR, staging_dir)
        
        # Display migration plan summary
        print(f"Migration plan created with {len(migration_plan)} tasks:")
        for i, task in enumerate(migration_plan[:5], 1):
            if task["action"] == "skip":
                print(f"  {i}. Skip {task['source']} - {task['reason']}")
            else:
                print(f"  {i}. {task['action']} from {task['source']} to {task['target']}")
        
        if len(migration_plan) > 5:
            print(f"  ... and {len(migration_plan) - 5} more tasks")
        
        # Step 4: Generate a migration script
        script_path = ROOT_DIR / "scripts" / "reorganize_repo.sh"
        if not (ROOT_DIR / "scripts").exists():
            (ROOT_DIR / "scripts").mkdir(parents=True, exist_ok=True)
        
        script_path = generate_migration_script(migration_plan, script_path)
        print(f"\n✅ Migration script generated at {script_path}")
        print("You can execute this script to perform the actual migration.")
        
        # Step 5: Final validation
        print("\nValidation complete. The new structure is ready to be applied.")
        print(f"Staging directory: {staging_dir}")
        print(f"Migration script: {script_path}")
"""

print("⚠️ The execution code is commented out for safety.")
print("To execute the reorganization:")
print("1. Remove the triple quotes around the code in this cell")
print("2. Run the cell again")
print("3. Review the results")
print("4. Execute the generated migration script when ready")

## Conclusion

This notebook provides a complete process for reorganizing the Orpheus Engine project according to the HP AI Studio Blueprints Repository format. By following these steps, you can:

1. Create a standardized directory structure
2. Generate template files for documentation and configuration
3. Migrate existing files to their new locations
4. Validate the resulting structure

The reorganization improves:

- **Maintainability**: Clear separation of concerns with a standardized structure
- **Discoverability**: Consistent file locations across projects
- **Documentation**: Comprehensive README files and documentation templates
- **Scalability**: Structure designed for growth and additional features

### Next Steps

After reorganizing the project structure:

1. Review the new structure and update any import paths in code files
2. Update build configurations to work with the new structure
3. Update documentation to reflect the new organization
4. Test the application to ensure everything works correctly

Remember that this is a significant change to the project structure, so it's important to test thoroughly and communicate the changes to all team members.